# Lesson 4: Build with a language model

From [Forward Deployed Engineering](https://gradlabs.in/learn/fde/lessons/lesson4.html). The client, Northstar Skills, is made up: every name, number and message is invented.

Make Northstar's admissions agent answer correctly: rules in a system prompt, answers your code can read, and tools that check the real fee sheet and calendar.

For each cell: **guess what it will print first**, then press ▶ and check.

**1. Get the code.**

In [ ]:
import os
if not os.path.exists("/content/forward-deployed-engineering"):
    !git clone -q --depth 1 https://github.com/mo-ri-ch/forward-deployed-engineering /content/forward-deployed-engineering
%cd /content/forward-deployed-engineering/starter/northstar-agent
!pip install -q google-genai anthropic openai

**2. Choose a model.** Start with `"replay"`: no key needed, and every step shows a recorded answer. For a live model, make a free Gemini key at [aistudio.google.com](https://aistudio.google.com), save it under 🔑 Secrets as `GEMINI_API_KEY`, and change `PROVIDER` to `"gemini"`.

Free tiers may use what you send to improve the provider's products. Northstar's data is made up, so that's fine here. Never send a real client's data to a free tier.

In [ ]:
PROVIDER = "replay"   # "gemini" (free tier), "claude", "openai", or "replay" (no key: recorded answers)

import llm
llm.setup(PROVIDER)

**3. A first call, with no rules.** A parent asks about the fee and a job guarantee.

🤔 *The model has never seen Northstar's fee sheet. What will it say?*

In [ ]:
MESSAGE = "hi wat is fees for full stack? job guranteed?"
print(llm.ask(MESSAGE, label="l4-no-rules"))

It guessed a fee, and may have promised a job. Both are things Northstar must never say.

**4. Write the rules: a system prompt.** Give it who it is, the real fees, and what it must never do.

In [ ]:
import northstar

SYSTEM = f"""You are the admissions assistant for Northstar Skills, a skills institute.
Only quote fees from this list:
{northstar.fee_table()}
Never promise a job. Say the placement team helps every student prepare and apply.
Reply in under 60 words, in the enquirer's language. Offer a free counselling call at the end."""

print(SYSTEM)

In [ ]:
print(llm.ask(MESSAGE, system=SYSTEM, label="l4-with-rules"))

🤔 *With a live model: delete the "Never promise a job" line, run both cells again, and see what changes.*

**5. Answers your code can use.** Ask for JSON, so your code can decide what to do. Five real-style messages; the last column is the answer you'd expect.

In [ ]:
CLASSIFY = """Read the enquiry. Reply with JSON only, like:
{"intent": "fees", "course": "Python & AI", "needs_human": false}
intent is one of: fees, booking, course_info, scholarship, refund, complaint, other.
course is a Northstar course name, or null.
needs_human is true for money trouble, grief, refunds, complaints, or when you are unsure."""

tests = [
    ("chetta full stack course nte fees ethra aanu", "fees", False),
    ("can I book a call saturday morning?", "booking", False),
    ("my father passed away, we can't pay full fees. any scholarship?", "scholarship", True),
    ("I joined last month and want a refund", "refund", True),
    ("is the digital marketing course online?", "course_info", False),
]

right = 0
for i, (text, want_intent, want_human) in enumerate(tests):
    out = llm.ask_json(text, system=CLASSIFY, label=f"l4-classify-{i}")
    ok = out.get("intent") == want_intent and out.get("needs_human") == want_human
    right += ok
    print("✓" if ok else "✗", text[:45].ljust(45), out)
print(f"\n{right} of {len(tests)} right")

That last loop is a tiny **eval set**: messages plus the answers you expect. You'll grow it in Lesson 7. Notice the last reply came wrapped in ```json fences: `ask_json` finds the JSON inside, because models often do that.

**6. What does it cost?** About ¾ of a word per token in English, so tokens ≈ words ÷ 0.75. With a live model, `llm.usage` also shows the real counts the provider reported.

🤔 *About how many tokens is the system prompt?*

In [ ]:
words = len(SYSTEM.split())
print(words, "words ≈", round(words / 0.75), "tokens")
print("Reported so far:", llm.usage)

PRICE_PER_MILLION = 2.00   # an example price in dollars. Check your model's real price.
tokens_per_enquiry = 1500
monthly = 2500 * tokens_per_enquiry / 1_000_000 * PRICE_PER_MILLION
print(f"2,500 enquiries a month ≈ ${monthly:.2f}")

**7. Tools.** The model can't see today's calendar, but your code can. Try the two tools directly first.

In [ ]:
print(northstar.get_fee("mern course"))
print(northstar.find_slots("2026-11-21", "morning"))

Now tell the model about the tools. When it needs one, it replies with a request; your code runs it and sends back the result. That loop is the whole idea behind tool use.

In [ ]:
import json

AGENT = SYSTEM + """

Today is Sunday 15 November 2026.
You can use these tools. To use one, reply with JSON only: {"tool": "get_fee", "args": {"course": "Python & AI"}}
- get_fee(course): the current fee and EMI for a course
- find_slots(day, part): free counselling slots. day is YYYY-MM-DD. part is morning, afternoon, evening or any.
When you have what you need, reply with JSON only: {"reply": "your message to the enquirer"}"""


def run_agent(message, label, max_steps=5):
    transcript = f"Enquirer: {message}"
    for step in range(max_steps):
        out = llm.ask_json(transcript, system=AGENT, label=f"{label}-{step}")
        if "reply" in out:
            print("Reply:", out["reply"])
            return
        name, args = out["tool"], out.get("args", {})
        result = northstar.TOOLS[name](**args)
        print(f"Step {step + 1}: {name}({args})\n   → {result}")
        transcript += f"\nTool {name} returned: {json.dumps(result, ensure_ascii=False)}"
    print("Stopped: too many steps.")

🤔 *Which tool will each message need?*

In [ ]:
run_agent("Any counselling slot on Saturday morning?", "l4-sat")

In [ ]:
run_agent("fees for python course? EMI possible?", "l4-py")

In [ ]:
run_agent("Python course fee, and can someone call me on Monday?", "l4-both")

The last one used two tools, so it took 3 model calls: one per tool, plus one to write the reply.

**8. Same question, different answers.** Ask three times at low and at high temperature.

🤔 *Which would you trust to quote a fee?*

Some newer models, like Claude's current ones, don't offer a temperature setting. With them, both rows behave the same.

In [ ]:
for temperature, label in [(0.0, "l4-temp-low"), (1.0, "l4-temp-high")]:
    print(f"Temperature {temperature}:")
    for _ in range(3):
        print("  ", llm.ask("fees for full stack? one line.", system=SYSTEM, temperature=temperature, label=label))

**9. Your turn.** Add a third tool, `hand_to_human(reason)`, that returns `{"handed_over": True, "reason": reason}`. Add it to `northstar.TOOLS` and to the tool list in `AGENT`, then run `run_agent("I joined last month and want a refund", "mine")` with a live model. Does it hand over?

In [ ]:
# Write your hand_to_human tool here


**10. Put it on your GitHub.** Save a copy of this notebook (File → Save a copy in GitHub), or download it and add it to your portfolio repo. Next lesson, the agent answers from Northstar's course pages.